In [1]:
import pyspark.sql.functions as f
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

In [2]:
spark.sparkContext.setLogLevel("ERROR")  # or "WARN"
spark

In [3]:
%%sql
SHOW CATALOGS

catalog
demo
spark_catalog


In [4]:
!ls /home/iceberg/data_sync/nba

games.csv  games_details.csv  players.csv  ranking.csv	teams.csv


In [5]:
%%sql
CREATE DATABASE IF NOT EXISTS demoTryo

++
||
++
++

In [6]:
%%sql
SHOW DATABASES

namespace
demoTryo


In [7]:
game_details = spark.read.csv("/home/iceberg/data_sync/nba/games_details.csv", header=True, inferSchema=True)
game_details.printSchema()

root
 |-- GAME_ID: integer (nullable = true)
 |-- TEAM_ID: integer (nullable = true)
 |-- TEAM_ABBREVIATION: string (nullable = true)
 |-- TEAM_CITY: string (nullable = true)
 |-- PLAYER_ID: integer (nullable = true)
 |-- PLAYER_NAME: string (nullable = true)
 |-- NICKNAME: string (nullable = true)
 |-- START_POSITION: string (nullable = true)
 |-- COMMENT: string (nullable = true)
 |-- MIN: string (nullable = true)
 |-- FGM: double (nullable = true)
 |-- FGA: double (nullable = true)
 |-- FG_PCT: double (nullable = true)
 |-- FG3M: double (nullable = true)
 |-- FG3A: double (nullable = true)
 |-- FG3_PCT: double (nullable = true)
 |-- FTM: double (nullable = true)
 |-- FTA: double (nullable = true)
 |-- FT_PCT: double (nullable = true)
 |-- OREB: double (nullable = true)
 |-- DREB: double (nullable = true)
 |-- REB: double (nullable = true)
 |-- AST: double (nullable = true)
 |-- STL: double (nullable = true)
 |-- BLK: double (nullable = true)
 |-- TO: double (nullable = true)
 |-- PF

In [8]:
games = spark.read.csv("/home/iceberg/data_sync/nba/games.csv", header=True, inferSchema=True)
games.printSchema()

root
 |-- GAME_DATE_EST: date (nullable = true)
 |-- GAME_ID: integer (nullable = true)
 |-- GAME_STATUS_TEXT: string (nullable = true)
 |-- HOME_TEAM_ID: integer (nullable = true)
 |-- VISITOR_TEAM_ID: integer (nullable = true)
 |-- SEASON: integer (nullable = true)
 |-- TEAM_ID_home: integer (nullable = true)
 |-- PTS_home: double (nullable = true)
 |-- FG_PCT_home: double (nullable = true)
 |-- FT_PCT_home: double (nullable = true)
 |-- FG3_PCT_home: double (nullable = true)
 |-- AST_home: double (nullable = true)
 |-- REB_home: double (nullable = true)
 |-- TEAM_ID_away: integer (nullable = true)
 |-- PTS_away: double (nullable = true)
 |-- FG_PCT_away: double (nullable = true)
 |-- FT_PCT_away: double (nullable = true)
 |-- FG3_PCT_away: double (nullable = true)
 |-- AST_away: double (nullable = true)
 |-- REB_away: double (nullable = true)
 |-- HOME_TEAM_WINS: integer (nullable = true)



In [9]:
games.writeTo("demoTryo.nba_games") \
  .using("iceberg") \
  .partitionedBy(f.col("SEASON")) \
  .createOrReplace()

In [10]:
%%sql
SELECT 
    *
FROM demoTryo.nba_games.snapshots

committed_at,snapshot_id,parent_id,operation,manifest_list,summary
2025-11-24 12:42:55.953000,2271842415557641635,None,append,s3://warehouse/demoTryo/nba_games/metadata/snap-2271842415557641635-1-c7d7c688-63fc-4220-9cf4-4b553eaecadd.avro,"{'engine-version': '3.5.5', 'added-data-files': '20', 'total-equality-deletes': '0', 'app-id': 'local-1763422686127', 'added-records': '26651', 'total-records': '26651', 'spark.app.id': 'local-1763422686127', 'changed-partition-count': '20', 'engine-name': 'spark', 'total-position-deletes': '0', 'added-files-size': '714904', 'total-delete-files': '0', 'iceberg-version': 'Apache Iceberg 1.8.1 (commit 9ce0fcf0af7becf25ad9fc996c3bad2afdcfd33d)', 'total-files-size': '714904', 'total-data-files': '20'}"
2025-11-24 12:45:50.953000,6964464412782636261,None,append,s3://warehouse/demoTryo/nba_games/metadata/snap-6964464412782636261-1-aae47d1f-42df-4906-87a2-8b49e8c2a0c8.avro,"{'engine-version': '3.5.5', 'added-data-files': '20', 'total-equality-deletes': '0', 'app-id': 'local-1763988345167', 'added-records': '26651', 'total-records': '26651', 'spark.app.id': 'local-1763988345167', 'changed-partition-count': '20', 'engine-name': 'spark', 'total-position-deletes': '0', 'added-files-size': '714904', 'total-delete-files': '0', 'iceberg-version': 'Apache Iceberg 1.8.1 (commit 9ce0fcf0af7becf25ad9fc996c3bad2afdcfd33d)', 'total-files-size': '714904', 'total-data-files': '20'}"


In [11]:
%%sql
SELECT *
FROM   demoTryo.nba_games.manifests

content,path,length,partition_spec_id,added_snapshot_id,added_data_files_count,existing_data_files_count,deleted_data_files_count,added_delete_files_count,existing_delete_files_count,deleted_delete_files_count,partition_summaries
0,s3://warehouse/demoTryo/nba_games/metadata/aae47d1f-42df-4906-87a2-8b49e8c2a0c8-m0.avro,12983,0,6964464412782636261,20,0,0,0,0,0,"[Row(contains_null=False, contains_nan=False, lower_bound='2003', upper_bound='2022')]"


In [12]:
%%sql
SELECT * from demoTryo.nba_games.partitions

partition,spec_id,record_count,file_count,total_data_file_size_in_bytes,position_delete_record_count,position_delete_file_count,equality_delete_record_count,equality_delete_file_count,last_updated_at,last_updated_snapshot_id
Row(SEASON=2022),0,542,1,22412,0,0,0,0,2025-11-24 12:45:50.953000,6964464412782636261
Row(SEASON=2004),0,1362,1,35864,0,0,0,0,2025-11-24 12:45:50.953000,6964464412782636261
Row(SEASON=2005),0,1432,1,37206,0,0,0,0,2025-11-24 12:45:50.953000,6964464412782636261
Row(SEASON=2003),0,1385,1,35584,0,0,0,0,2025-11-24 12:45:50.953000,6964464412782636261
Row(SEASON=2008),0,1425,1,37014,0,0,0,0,2025-11-24 12:45:50.953000,6964464412782636261
Row(SEASON=2009),0,1424,1,37120,0,0,0,0,2025-11-24 12:45:50.953000,6964464412782636261
Row(SEASON=2006),0,1419,1,37300,0,0,0,0,2025-11-24 12:45:50.953000,6964464412782636261
Row(SEASON=2007),0,1411,1,37130,0,0,0,0,2025-11-24 12:45:50.953000,6964464412782636261
Row(SEASON=2012),0,1420,1,37007,0,0,0,0,2025-11-24 12:45:50.953000,6964464412782636261
Row(SEASON=2013),0,1427,1,37337,0,0,0,0,2025-11-24 12:45:50.953000,6964464412782636261
